In [ ]:
import os
import pandas as pd
import numpy as np
import re
import requests

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# function to get unique values
def unique(list1):
 
    # initialize a null list
    unique_list = []
 
    # traverse for all elements
    for x in list1:
        # check if exists in unique_list or not
        if x not in unique_list:
            unique_list.append(x)
    return unique_list


In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
path_git  = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
path_config = os.path.join(path_git, 'config')


In [ ]:
# Import Area Codes
df_codes = pd.read_excel(os.path.join(path_config, 'Area Codes.xlsx'), sheet_name = 'CountyFIPS')
df_codes = df_codes[df_codes['State'] == 'CA'].dropna()
df_codes.head()

In [ ]:
## Population Estimates
## E4
# 2020-2024
# 2010-2020
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2024_InternetVersion.xlsx"
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-4_2010-2020-Internet-Version.xlsx"
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E4_2000-2010_Report_Final_EOC_000.xlsx"

## Population and Housing Estimates
## E5
# 2020-2024
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
## E8
# 2010-2020
# 2000-2010
url = "https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx"
url = "https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"

# Household population vs Total population?

In [ ]:

## Import DOF data

# Set browser user agent
headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

dict_url = {
    2020:"https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/E-5-2024_Geo_InternetVersion.xlsx"
    , 2010:"https://dof.ca.gov/wp-content/uploads/sites/352/2023/11/E-8_2010_2020_by_Geo_Internet.xlsx"
    , 2000:"https://dof.ca.gov/wp-content/uploads/sites/352/Forecasting/Demographics/Documents/Closed_E8_Full_Decade_Final_v2.xlsx"
}

list_df_state    = []
list_df_counties = []
list_df_cities   = []
list_df_balance  = []

for year in list(dict_url.keys()):
    if year == 2000:
        request = requests.get(dict_url[year], headers = headers)
        request = request.content
        df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
        df = df.dropna(subset = ['Household'])
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        
        # Initialize 'County' and 'City' columns
        df['County'] = np.nan
        df['City'  ] = np.nan
        
        county_temp = None
        
        # Reset index
        df.reset_index(drop=True, inplace=True)
        
        for i in range(len(df)):
            if pd.notnull(df.loc[i, 'County / City']):
                if county_temp is not None:
                    df.loc[i, 'County'] = county_temp
                    df.loc[i, 'City'] = df.loc[i, 'County / City']
                    county_temp = None
                else:
                    county_temp = df.loc[i, 'County / City']
        
        df['County'].fillna(method='ffill', inplace = True)
        df['City'  ].fillna(method='ffill', inplace = True)
        
        # shift up and then fill the last row
        df['County'] = df['County'].shift(-1)
        df['City'  ] = df['City'  ].shift(-1)
        
        df['County'].fillna(method = 'ffill', inplace = True)
        df['City'  ].fillna(method = 'ffill', inplace = True)
        df = df.drop(columns = ['County / City'])

        df = df[df['Year'] != 2010]
        
        df = df[['County', 'City', 'Year', 'Total', 'Household', 'Total.1', 'Occupied']].rename(columns = {'Household': 'Household Population'
                                                                                                           , 'Total'  : 'Population'
                                                                                                           , 'Total.1': 'Housing Units'})
        df['Unoccupied'] = df['Housing Units'] - df['Occupied']

        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        df = df[df['County'].isin(df_codes['County Name'].values)]
        df = df.merge(df_codes[['County Name', 'MPO']], left_on = 'County', right_on = 'County Name', how = 'left').drop('County Name', axis = 1)
        
        # Subset
        df_sf = df[df['County'] == 'San Francisco']
        df_sf.loc[:, 'City'] = 'County Total'
        df = pd.concat([df, df_sf])
        df_cities   = df[~df['City'].isin(['County Total', 'Incorporated', 'Balance of County'])].reset_index(drop = True).drop('County', axis = 1)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )

    else:
        request = requests.get(dict_url[year], headers = headers)
        request = request.content
        df = pd.read_excel(request, sheet_name = 1, skiprows = 2, engine='openpyxl')
        df['Date'] = pd.to_datetime(df['Date'])
        df['Year'] = df['Date'].dt.year
        if year == 2010:
            df = df[df['Year'] != 2020]

        
        df = df[['County', 'City', 'Year', 'Total', 'Household', 'Total.1', 'Occupied']].rename(columns = {'Household': 'Household Population'
                                                                                                           , 'Total'  : 'Population'
                                                                                                           , 'Total.1': 'Housing Units'})
        df['Unoccupied'] = df['Housing Units'] - df['Occupied']
        
        df_state = df[df['County'] == 'California'].rename(columns = {'County':'State'})
        df       = df[df['County'] != 'California']
        df = df[df['County'].isin(df_codes['County Name'].values)]
        df = df.merge(df_codes[['County Name', 'MPO']], left_on = 'County', right_on = 'County Name', how = 'left').drop('County Name', axis = 1)
        
        # Subset
        if year == 2020:
            df_sf = df[df['County'] == 'San Francisco']
            df_sf.loc[:, 'City'] = 'San Francisco'
            df = pd.concat([df, df_sf])

        if year == 2010:
            df_sf = df[df['County'] == 'San Francisco']
            df_sf.loc[:, 'City'] = 'County Total'
            df = pd.concat([df, df_sf])
            
        df_cities   = df[~df['City'].isin(['County Total', 'Incorporated', 'Balance of County'])].reset_index(drop = True).drop('County', axis = 1)
        df_counties = df[ df['City'].str.contains('County Total'     )].reset_index(drop = True).drop('City'  , axis = 1)
        df_balance  = df[ df['City'].str.contains('Balance of County')].reset_index(drop = True).drop('City'  , axis = 1)
        
        list_df_state   .append(df_state   )
        list_df_counties.append(df_counties)
        list_df_cities  .append(df_cities  )
        list_df_balance .append(df_balance )
            

df_state    = pd.concat(list_df_state   )
df_counties = pd.concat(list_df_counties)
df_cities   = pd.concat(list_df_cities  )
df_balance  = pd.concat(list_df_balance )

df_state    = df_state   .sort_values(['City'  , 'Year'], ascending = [True, False])
df_counties = df_counties.sort_values(['County', 'Year'], ascending = [True, False])
df_cities   = df_cities  .sort_values(['City'  , 'Year'], ascending = [True, False])
df_balance  = df_balance .sort_values(['County', 'Year'], ascending = [True, False])

print('By State')
print(df_state   .head())
print('By Counties')
print(df_counties.head())
print('By Cities')
print(df_cities  .head())
print('By Balance')
print(df_balance .head())

***

## Pop_1

***

In [ ]:
# Pop_1
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', 'Pop_1')

df_pop1 = df_counties[['County', 'Year', 'Population', 'Household Population']].drop_duplicates()
df_pop1.to_excel(os.path.join(path_out, 'Pop_1_DOF_County.xlsx'), index = False)

In [ ]:
path_plots = os.path.join(path_out, 'plots')

df_plot = df_pop1.copy()


x = 'Year'
y = 'Population'
color = 'County'
labels = 'County'


fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = True
                 , labels = labels
                )

fig.update_layout(title = 'Population by County')

fig.write_html(
    os.path.join(
        path_plots
        , ''.join(['Pop_1' + '_'
                   , 'Population by County_'
                   , 'line_'
                   , '.html'])
    )
)
    

fig.show()

***

## Pop_2

***

In [ ]:
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'People and Community', 'Pop and Demographics', 'Pop_2')

# Jurisdiction
df_pop2 = df_cities[['City', 'Year', 'Population', 'Household Population']]
df_pop2 = df_pop2[df_pop2['City'] != 'Incorporated']
df_pop2 = df_pop2.sort_values(['City', 'Year'], ascending = [True, True])
df_pop2['Population_GR'          ] = df_pop2['Population'          ].pct_change()
df_pop2['Household Population_GR'] = df_pop2['Household Population'].pct_change()
df_pop2.loc[df_pop2['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop2.loc[df_pop2['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop2 = df_pop2.sort_values(['City', 'Year'], ascending = [True, False])

df_pop2.to_excel(os.path.join(path_out, 'Pop_2_DOF_Jurisdiction.xlsx'), index = False)

# County
df_pop2 = df_counties[['County', 'Year', 'Population', 'Household Population']]
df_pop2 = df_pop2[df_pop2['County'] != 'Incorporated']
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, True])
df_pop2['Population_GR'          ] = df_pop2['Population'          ].pct_change()
df_pop2['Household Population_GR'] = df_pop2['Household Population'].pct_change()
df_pop2.loc[df_pop2['Year'] == 2000, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Year'] == 2000, 'Household Population_GR'] = np.nan
df_pop2.loc[df_pop2['Population_GR'          ] == np.inf, 'Population_GR'          ] = np.nan
df_pop2.loc[df_pop2['Household Population_GR'] == np.inf, 'Household Population_GR'] = np.nan
df_pop2 = df_pop2.sort_values(['County', 'Year'], ascending = [True, False])

df_pop2.to_excel(os.path.join(path_out, 'Pop_2_DOF_County.xlsx'), index = False)

In [ ]:
path_plots = os.path.join(path_out, 'plots')

df_plot = df_pop2.copy()
df_plot['Population_GR'] = round(df_plot['Population_GR']*100, 1)


x = 'Year'
y = 'Population_GR'
color = 'County'
labels = 'County'


fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = True
                 , labels = labels
                )

fig.update_layout(title = 'Population Growth Rate by County (%)')

fig.write_html(
    os.path.join(
        path_plots
        , ''.join(['Pop_2' + '_'
                   , 'Population Growth Rate by County_'
                   , 'line_'
                   , '.html'])
    )
)
    

fig.show()

***

## Cost_3

***

In [ ]:
# Cost_3
path_out = os.path.join(path_main, 'Vibrant and Inclusive Places', 'Development', 'Housing Cost', 'Cost_3')

# Juridiction
df_cost3 = df_cities[['City', 'Year', 'Housing Units', 'Occupied', 'Unoccupied']]
df_cost3 = df_cost3.sort_values(['City', 'Year'], ascending = [True, True])
df_cost3['Housing Units_GR'] = df_cost3['Housing Units'].pct_change()
df_cost3['Occupied_GR'     ] = df_cost3['Occupied'     ].pct_change()
df_cost3['Unoccupied_GR'   ] = df_cost3['Unoccupied'   ].pct_change()
df_cost3.loc[df_cost3['Year'] == 2000, 'Housing Units_GR'] = np.nan
df_cost3.loc[df_cost3['Year'] == 2000, 'Occupied_GR'     ] = np.nan
df_cost3.loc[df_cost3['Year'] == 2000, 'Unoccupied_GR'   ] = np.nan
df_cost3 = df_cost3.sort_values(['City', 'Year'], ascending = [True, False])

df_cost3.to_excel(os.path.join(path_out, 'Cost_3_DOF_Jurisdiction.xlsx'), index = False)

# County
df_cost3 = df_counties[['County', 'Year', 'Housing Units', 'Occupied', 'Unoccupied']]
df_cost3 = df_cost3.sort_values(['County', 'Year'], ascending = [True, True])
df_cost3['Housing Units_GR'] = df_cost3['Housing Units'].pct_change()
df_cost3['Occupied_GR'     ] = df_cost3['Occupied'     ].pct_change()
df_cost3['Unoccupied_GR'   ] = df_cost3['Unoccupied'   ].pct_change()
df_cost3.loc[df_cost3['Year'] == 2000, 'Housing Units_GR'] = np.nan
df_cost3.loc[df_cost3['Year'] == 2000, 'Occupied_GR'     ] = np.nan
df_cost3.loc[df_cost3['Year'] == 2000, 'Unoccupied_GR'   ] = np.nan
df_cost3 = df_cost3.sort_values(['County', 'Year'], ascending = [True, False])

df_cost3.to_excel(os.path.join(path_out, 'Cost_3_DOF_County.xlsx'), index = False)

In [ ]:
# Cost_3
path_plots = os.path.join(path_out, 'plots')

df_plot = df_cost3.copy()
df_plot['Unoccupied_GR'] = round(df_plot['Unoccupied_GR']*100, 1)
df_plot = df_plot[df_plot['County'] != 'San Benito'] # i think there is a data quality issue with san benito

x = 'Year'
y = 'Unoccupied_GR'
color = 'County'
labels = 'County'


fig = px.line(df_plot
                 , x = x
                 , y = y
                 , color = color
                 # , line_dash = line_dash
                 , markers = True
                 , labels = labels
                )

fig.update_layout(title = 'Unoccupied Housing Growth Rate by County (%)')

fig.write_html(
    os.path.join(
        path_plots
        , ''.join(['Cost_3' + '_'
                   , 'Unoccupied Housing Growth Growth Rate by County_'
                   , 'line_'
                   , '.html'])
    )
)
    

fig.show()

Code graveyard

In [ ]:
# header = {
#   "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
#   "X-Requested-With": "XMLHttpRequest"
# }

# r = requests.get(site, headers=header)
# r.text

In [ ]:
# import pandas as pd
# import requests

# # Check the end of the url -->                                                                             HERE --v
# url = 'https://<myOrg>.sharepoint.com/:x:/s/x-taulukot/Ec0R1y3l7sdGsP92csSO-mgBI8WCN153LfEMvzKMSg1Zzg?e=6NS5Qh&download=1'
# headers = {'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/114.0'}

# resp = requests.get(url, headers=headers)
# df = pd.read_excel(resp.content, engine='openpyxl')